In [ ]:
#UCI Primary Tumor
import autosklearn.classification
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import sklearn.metrics

In [ ]:
table = pd.read_table('../Datasets/Primary Tumor/primary-tumor.data', sep=',', header=None)

In [ ]:
table

In [ ]:
columns = ['class', 'age', 'sex', 'histologic-type', 'degree-of-diffe', 'bone', 'bone-marrow', 'lung', 'pleura', 'peritoneum', 'liver', 'brain', 'skin', 'neck', 'supraclavicular', 'axillar', 'mediastinum', 'abdominal']

In [ ]:
table = table.rename(columns={i: columns[i] for i in range(len(columns))})

In [ ]:
table.replace('?', np.nan, inplace=True)

In [ ]:
table = table.astype(float)

In [ ]:
table = table - 1

In [ ]:
table.drop('degree-of-diffe', axis=1, inplace=True)

In [ ]:
table.dropna(inplace=True)

In [ ]:
table = table.astype(int)

In [ ]:
X, y = table.drop('class', axis=1), table['class']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# AUTOML

In [ ]:
automl = autosklearn.classification.AutoSklearnClassifier(time_left_for_this_task=600)
automl.fit(X_train, y_train)
y_hat = automl.predict(X_test)

In [ ]:
sklearn.metrics.accuracy_score(y_test, y_hat)

In [ ]:
import pickle
with open('./models/primary_tumor_automl2.pkl', 'wb') as f:
    pickle.dump(automl, f)

In [ ]:
import pickle
with open('./models/primary_tumor_automl2.pkl', 'rb') as f:
    automl = pickle.load(f)

len(automl.show_models())

# RANDOM FOREST

In [ ]:
import sklearn.ensemble

model = sklearn.ensemble.RandomForestClassifier(n_estimators=100, n_jobs=5, random_state=42)
model.fit(X_train, y_train)
y_hat = model.predict(X_test)

In [ ]:
sklearn.metrics.accuracy_score(y_test, y_hat)

In [ ]:
import pickle
with open('./models/primary_tumor_rf.pkl', 'wb') as f:
    pickle.dump(model, f)

# SVC

In [ ]:
import sklearn.svm
import sklearn.model_selection

model = sklearn.svm.SVC(random_state=42, probability=True)
# params = {'C': [0.1, 1, 10, 100, 1000], 'gamma': [1, 0.1, 0.01, 0.001, 0.0001], 'kernel': ['linear', 'ploy', 'rbf'], 'degree': [3, 4, 5]}
# model = sklearn.model_selection.GridSearchCV(svc, param_grid=params, cv=5, n_jobs=5, verbose=3)
model.fit(X_train, y_train)
y_hat = model.predict(X_test)
sklearn.metrics.accuracy_score(y_test, y_hat)

In [ ]:
import pickle
with open('./models/primary_tumor_svm.pkl', 'wb') as f:
    pickle.dump(model, f)

# LIME

In [ ]:
import lime
import lime.lime_tabular
import tqdm 

In [ ]:
categorical_features = X_train.columns.tolist()
explainer = lime.lime_tabular.LimeTabularExplainer(X_train.values, feature_names=X_train.columns.tolist(), class_names=[str(i) for i in range(22)], categorical_features=categorical_features, discretize_continuous=True)

In [ ]:
text_x = X_test.values

In [ ]:
exp_fn = lambda i: explainer.explain_instance(X_test.iloc[i], model.predict_proba, num_features=len(X_test.columns))
def exp_fn_blk(xtest, exp_fn):
    exp1 = []
    for i in tqdm.tqdm(range(len(xtest))):
        exp = exp_fn(i)
        exp1.append(exp.as_map()[exp.available_labels()[0]])
    return np.array(exp1)
exp_fn_wrap = lambda x: np.array(exp_fn_blk(x, exp_fn))

In [ ]:
import metrics

In [ ]:
exp1 = exp_fn_wrap(text_x)
exp2 = exp_fn_wrap(text_x)

In [ ]:
np.save('./explanations/primaryTumor1.npy', exp1)
np.save('./explanations/primaryTumor2.npy', exp2)

In [ ]:
def enc_exp(exp, feature_num):
    enc_exp = np.zeros((len(exp),feature_num))
    for i in range(len(exp)):
        for j in range(len(exp[i])):
            enc_exp[i][int(exp[i,j,0])] = exp[i,j,1]
    return enc_exp

In [ ]:
i = metrics.calc_identity(exp1, exp2)
s = metrics.calc_separability(exp1)
enc1 = enc_exp(exp1, len(X_test.columns))
sb = metrics.calc_stability(enc1, y_test)

In [ ]:
i, s, sb

In [ ]:
X_test_norm = metrics.normalize_test(X_train, X_test)
sim = metrics.calc_similarity(exp1, X_test_norm)

In [ ]:
sim

In [ ]:
sort_uni_y = np.unique(y_test)

In [ ]:
list_monotonicity = []
list_non_sensitivity = []
list_effective_complexity = []

for i in tqdm.tqdm(range(len(text_x))):
    atr = exp1[i]
    sorted_atr = [j for i,j in atr]
    sorted_feat = [i for i,j in atr]
    y = np.zeros(19, dtype=int)
    np.put(y, np.where(sort_uni_y == y_test.iloc[i])[0][0], 1)
    example = metrics.FeatureAttribution(model, text_x[i], y, sorted_atr)
    list_monotonicity.append(example.monotonicity())
    list_non_sensitivity.append(example.non_sensitivity())
    list_effective_complexity.append(example.effective_complexity(sorted_feat, 0.1))

In [ ]:
print(np.mean(list_monotonicity))
print(np.mean(list_non_sensitivity))
print(np.mean(list_effective_complexity))

print(np.median(list_monotonicity))
print(np.median(list_non_sensitivity))
print(np.median(list_effective_complexity))

In [ ]:
trust = metrics.calc_trust_score(model, text_x, exp1, 3, X_train.columns.tolist())

In [ ]:
trust

# CIU

In [ ]:
from ciu import determine_ciu
import tqdm
import metrics

In [ ]:
def enc_exp(exp, feature_num):
    enc_exp = np.zeros((len(exp),feature_num))
    for i in range(len(exp)):
        for j in range(len(exp[i])):
            enc_exp[i][int(exp[i,j,0])] = exp[i,j,1]
    return enc_exp

In [ ]:
feat_list = X_train.columns.tolist()

In [ ]:
def exp_fn_blk(xtest):
    exp1 = []
    for i in tqdm.tqdm(range(len(xtest))):
        exp = determine_ciu(X_test.iloc[i:i+1], model.predict_proba, X_train.to_dict('list'), samples = 1000, prediction_index = 1)
        exp_list = [[feat_list.index(i), exp.ci[i]] for i in exp.ci]
        exp1.append(exp_list)
    return np.array(exp1)

In [ ]:
exp1 = exp_fn_blk(X_test)
exp2 = exp_fn_blk(X_test)

In [ ]:
np.save('./explanations/primaryTumor_ciu1.npy', exp1)
np.save('./explanations/primaryTumor_ciu2.npy', exp2)

In [ ]:
i = metrics.calc_identity(exp1, exp2)
s = metrics.calc_separability(exp1)
enc1 = enc_exp(exp1, len(feat_list))
sb = metrics.calc_stability(enc1, y_test)

In [ ]:
i, s, sb

In [ ]:
X_test_norm = metrics.normalize_test(X_train, X_test)
sim = metrics.calc_similarity(exp1, X_test_norm)

In [ ]:
sim

In [ ]:
list_monotonicity = []
list_non_sensitivity = []
list_effective_complexity = []

for i in tqdm.tqdm(range(len(X_test))):
    atr = exp1[i]
    sorted_atr = [j for i,j in atr]
    sorted_feat = [i for i,j in atr]
    y = np.zeros(19, dtype=int)
    np.put(y, np.where(sort_uni_y == y_test.iloc[i])[0][0], 1)
    example = metrics.FeatureAttribution(model, X_test.to_numpy()[i], y, sorted_atr)
    list_monotonicity.append(example.monotonicity())
    list_non_sensitivity.append(example.non_sensitivity())
    list_effective_complexity.append(example.effective_complexity(sorted_feat, 0.1))

In [ ]:
print(np.mean(list_monotonicity))
print(np.mean(list_non_sensitivity))
print(np.mean(list_effective_complexity))

print(np.median(list_monotonicity))
print(np.median(list_non_sensitivity))
print(np.median(list_effective_complexity))

In [ ]:
metrics.calc_trust_score(model, X_test.to_numpy(), exp1, 3, X_train.columns.to_list())

# RULEFIT

In [ ]:
from skrules import SkopeRules
import metrics_rules
import time

In [ ]:
clf = SkopeRules(max_depth_duplication=2,
                    n_estimators=512,
                    precision_min=0.3,
                    recall_min=0.1,
                    feature_names=X_train.columns.tolist())

In [ ]:
start_time = time.time()
clf.fit(X_train, y_train)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
top_rules1 = clf.score_top_rules(X_test)
top_rules2 = clf.score_top_rules(X_test)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
i = metrics_rules.calc_identity_rules(top_rules1, top_rules2)
print(i)

s = metrics_rules.calc_separability_rules(top_rules1)
print(s)

enc_rules = metrics_rules.exp_enc(clf, top_rules1)
sb = metrics_rules.calc_stability_rules(enc_rules, y_test)
print(sb)

In [ ]:
X_test_norm = metrics_rules.normalize_test(X_train, X_test)
sim = metrics_rules.calc_similarity(enc_rules, X_test_norm)
print(sim)

# RULEMATRIX

In [ ]:
categorical_features = X_train.columns.tolist()
continuous_features = X_train.columns.drop(categorical_features).tolist()

In [ ]:
import rulematrix
import time
import metrics_rules

In [ ]:
is_continuous = [True if i in continuous_features else False for i in X_train.columns.tolist()]
is_categorical = [True if i in categorical_features else False for i in X_train.columns.tolist()]

In [ ]:
surrogate = rulematrix.surrogate.rule_surrogate(
    model.predict,
    X_train,
    sampling_rate=4,
    is_continuous=is_continuous,
    is_categorical=is_categorical,
    seed=42
)

In [ ]:
test_x = X_test.to_numpy()

In [ ]:
def exp_fn_blk(xtest):
    exp1 = []
    for i in range(len(xtest)):
        queried_rules = np.arange(surrogate.student.n_rules)[surrogate.student.decision_path(test_x[i].reshape(1,-1)).reshape(-1)]
        exp1.append(queried_rules[-1])
    return np.array(exp1)
exp_fn_wrap = lambda x: np.array(exp_fn_blk(x))

In [ ]:
start_time = time.time()
exp1 = exp_fn_blk(test_x)
exp2 = exp_fn_blk(test_x)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
def enc_exp(exp, n_features):
    enc = []
    for i in range(exp.shape[0]):
        new = np.zeros(n_features)
        for j in surrogate.student.rule_list[exp[i]].clauses:
            new[j.feature_idx] = 1
        enc.append(new)
    return np.array(enc)

In [ ]:
enc_exp = enc_exp(exp1, X_train.shape[1])

In [ ]:
i = metrics_rules.calc_identity_rules(exp1, exp2)
print(i)

s = metrics_rules.calc_separability_rules(exp1)
print(s)

sb = metrics_rules.calc_stability_rules(enc_exp, y_test)
print(sb)

In [ ]:
X_test_norm = metrics_rules.normalize_test(X_train, X_test)
sim = metrics_rules.calc_similarity(enc_exp, X_test_norm)

In [ ]:
sim